# A magpylib scene in Jupyter

The studio's 3D view in a notebook: the three.js renderer of the VS Code panel
and of `marimo_demo.py`, as an `anywidget`. Run it with this repo's `.venv` as
the kernel — in VS Code, pick that interpreter; for JupyterLab,
`uv pip install jupyterlab` into it and `jupyter lab examples/jupyter_demo.ipynb`.

Put the pointer on a view and its tools show in the top-right corner; the keys
button lists every key. Click a view to give it the keys: **F** frames the
selection, **1** **3** **7** look along the axes, **H** hides, **space** plays.

In [ ]:
import ipywidgets as widgets
import magpylib as magpy
import numpy as np

from magpylib_studio.widget import SceneWidget

## Sliders that rebuild the scene

Jupyter does not re-run a cell when a slider moves, so each slider rebuilds the
rings and hands them to the view with `update()`. The view is made once and
re-pointed, so the camera stays where you left it.

The legend is the scene as it is nested — the stack, its rings, their magnets.
Fold a ring with its caret, hide it with the eye, click a row to select what is
under it (⌘/ctrl adds, shift takes a range), double-click to frame it. The
selection is `scene.selected`, and `scene.picked` gives back the magpylib
objects; the readout under the view follows it.

In VS Code this view stays light in a dark theme, on purpose: it shares its
output with ipywidgets' sliders, which are drawn for white. A view alone in its
output follows VS Code's theme.

The sliders own this scene, so the view is made with `editable=None`: a view for
looking only, with no pencil, since an edit would be lost at the next move of a
slider. A view that edits is further down.


In [ ]:
def build(count, radius, tilt):
    """Two rings of magnets, turned opposite ways, and a probe in the middle."""

    def ring(name, z, turn):
        ring = magpy.Collection(style_label=f"{name} ring")
        for i in range(count):
            magnet = magpy.magnet.Cuboid(
                dimension=(1, 1, 1),
                polarization=(0, 0, 1),
                position=(radius, 0, z),
                style_label=f"{name} {i + 1}",
            )
            # Turned in place by the same angle for every magnet -- 0 degrees
            # is an axial ring, 90 a radial one -- then carried round.
            magnet.rotate_from_angax(turn, "y", anchor=magnet.position)
            magnet.rotate_from_angax(360 * i / count, "z", anchor=(0, 0, 0))
            ring.add(magnet)
        return ring

    stack = magpy.Collection(
        ring("upper", 1.0, tilt), ring("lower", -1.0, -tilt), style_label="stack"
    )
    return stack, magpy.Sensor(style_label="centre probe")

In [ ]:
count = widgets.IntSlider(6, min=2, max=16, description="magnets")
radius = widgets.FloatSlider(4.0, min=2.0, max=8.0, step=0.5, description="ring radius")
tilt = widgets.IntSlider(45, min=0, max=180, step=15, description="tilt °")
readout = widgets.HTML()
scene = SceneWidget(height=460, editable=None)  # the sliders own this scene
current = {}


def say():
    """The field at the probe, and what is selected, under the view."""
    field = np.round(magpy.getB(current["stack"], current["probe"]) * 1000, 3)
    picked = ", ".join(obj.style.label for obj in scene.picked) or "<i>nothing</i>"
    readout.value = f"<b>B at the probe:</b> {field} mT<br><b>Selected:</b> {picked}"


def rebuild(_=None):
    current["stack"], current["probe"] = build(count.value, radius.value, tilt.value)
    scene.update(current["stack"], current["probe"])
    say()


for slider in (count, radius, tilt):
    slider.observe(rebuild, names="value")
scene.observe(lambda _: say(), names="selected")
rebuild()

widgets.VBox([widgets.HBox([count, radius, tilt]), scene, readout])

## The magpylib way

`magpy.show(..., backend="widget")` draws the same view through magpylib's own
call, legend included, and `magpy.defaults.display.backend = "widget"` makes
every `show()` in the notebook draw one. Where sliders rebuild the scene, make
the view once with `SceneWidget` and `update()` it, as above: a `show()` that
runs again draws a new view.

In [ ]:
stack, probe = build(8, 3.0, 30)
magpy.show(stack, probe, backend="widget", height=380)

## Paths, played in the browser

`animation=True` captures every step of the run. Here every step is the first
one moved — a sensor sweeping past a magnet — so the view is handed the motion
rather than the frames: **▶** plays and the slider scrubs with nothing to ask
Python, which is why the run still plays in a saved page, or in a notebook read
without its kernel.

In [ ]:
rotor = magpy.magnet.Cylinder(
    polarization=(1, 0, 0), dimension=(4, 1), style_label="rotor"
)
sweep = magpy.Sensor(
    position=np.linspace((-6, 0, 1.5), (6, 0, 1.5), 60), style_label="sweep"
)
sweep.style.arrows.x.show = True
sweep.style.arrows.y.show = True

SceneWidget(rotor, sweep, animation=True, height=380)

## Properties along a path

On magpylib main, more than position and orientation can vary along a path —
`polarization`, `dimension`, `diameter`, `current` and others. Here a magnet's
polarization turns a full circle, a coil grows while its current ramps up, and
the probe's pixels show the field they sit in. What changes travels with the
view, so this too plays without Python.

In [ ]:
steps = 40
turn = np.linspace(0, 2 * np.pi, steps)
spinner = magpy.magnet.Cylinder(
    polarization=np.column_stack([np.cos(turn), np.sin(turn), np.zeros(steps)]),
    dimension=(2, 1),
    style_label="spinner",
)
coil = magpy.current.Circle(
    diameter=np.linspace(2, 5, steps),
    current=np.linspace(1, 100, steps),
    position=(0, 0, 1.5),
    style_label="coil",
)
probe = magpy.Sensor(
    pixel=np.linspace((-3, 0, 0), (3, 0, 0), 13),
    position=(0, 0, -1.5),
    style_label="probe",
)
probe.style.pixel.field.source = "B"  # each pixel an arrow of the field there

SceneWidget(spinner, coil, probe, animation=True, height=380)

## Keeping a view

The export icon saves a view as one HTML file that opens anywhere with no
notebook behind it — legend, keys, the camera where you left it, and a run that
still plays. `write_html` does the same from a cell. The camera icon saves a PNG
of the view as it is on screen.

In [ ]:
# scene.write_html("rings.html")  # the sliders' scene, as a file

## Editing the scene

`editable=True` puts out the studio's handles. Select an object and drag it —
**W** moves, **E** turns, **R** resizes, **P** aims, **L** switches between the
world's axes and the object's own — or use the column down the right. The
corner shows the numbers a drag changes and takes typed ones, and ⌘Z / ctrl-Z
undoes a whole drag. The objects are copied into a studio session in this
kernel, so the cell that made them stays as it was.

The log under the view shows what the view hands the notebook: a click
selects, the eye in the legend hides, and a drag, an undo or a `set` is an
edit — `last_edit` says what it was, and `revision` counts them, once per
finished drag.

In [ ]:
rotor = magpy.magnet.Cuboid(dimension=(1, 1, 1), polarization=(0, 0, 1))
pickup = magpy.Sensor(position=(0, 0, 2))
studio = SceneWidget(rotor, pickup, editable=True, height=380)
log = widgets.Output(layout={"max_height": "12em", "overflow": "auto"})


def logged(change):
    name = "edit" if change["name"] == "revision" else change["name"]
    what = studio.last_edit if name == "edit" else change["new"]
    # appended rather than printed inside `with log:`, which VS Code can
    # show several times over when a callback prints
    log.append_stdout(f"{name}: {what}\n")


studio.observe(logged, ["selected", "hidden", "revision"])
widgets.VBox([studio, log])

## A slider and the view, following each other

`set` edits an object from code as a drag does, one step to undo. With
`revision`, a slider and the view follow each other: the slider moves the pickup
in the view above, and dragging the pickup moves the slider. Neither echoes the
other — a value the pickup already has is no edit — and a drag past the
slider's end widens it rather than being clamped back.

In [ ]:
height = widgets.FloatSlider(
    min=0.5, max=4, step=0.05, value=2, description="pickup z", continuous_update=False
)


def from_slider(change):
    x, y, _ = studio.objects["pickup"].position
    studio.set("pickup", position=(x, y, change["new"]))


def from_view(change):
    z = float(studio.objects["pickup"].position[2])
    height.min, height.max = min(height.min, z), max(height.max, z)
    height.value = z


height.observe(from_slider, "value")
studio.observe(from_view, "revision")
height

## What was edited

`studio.objects` are the objects as edited — compute with them as with any
magpylib objects — and `to_script()` writes the scene, edits and all, as
magpylib code.

In [ ]:
pickup, rotor = studio.objects["pickup"], studio.objects["rotor"]
print("pickup at", pickup.position, "reads B =", pickup.getB(rotor), "T")
print(studio.to_script())

## The scene's variables

A scene written as a function has its parameters as variables. A view built from
it is a studio session from the start: the sliders toggle at the foot of its edit
column opens a slider per variable, and `variables` holds them as a trait a
notebook control is tied to, both ways. The ipywidgets slider below and the
panel's `n` move together, and the view owns the number; `to_script()` writes the
scene as it stands.


In [ ]:
from typing import Annotated

from magpylib_studio import Bounds, Count, duplicate_around, name, scene


@scene
def ring(
    n: Annotated[int, Count(3, 24, slider=(4, 16))] = 8,
    radius: Annotated[float, Bounds(2.0, 8.0)] = 4.0,
):
    """A ring of magnets: one magnet and one pattern."""
    magnet = name(
        magpy.magnet.Cuboid(
            dimension=(1, 1, 1), polarization=(0, 0, 1), position=(radius, 0, 0)
        ),
        "magnet",
    )
    group = name(magpy.Collection(magnet, style_label="ring"), "ring")
    duplicate_around(magnet, count=n, axis="z", spin=360 / n)
    return group


parametric = SceneWidget(ring.build(), editable=True, height=380)
magnets = widgets.IntSlider(8, min=3, max=24, description="magnets")
# Tied both ways: a value the scene already has is no edit, so neither side
# echoes the other.
magnets.observe(
    lambda change: setattr(parametric, "variables", {"n": change["new"]}), "value"
)
parametric.observe(
    lambda change: setattr(magnets, "value", int(change["new"]["n"])), "variables"
)
widgets.VBox([magnets, parametric])